# Notebook 5: Model Registry and Export

Pick the **best ALS run** (lowest **RMSE**) from Notebook 4’s experiment, **register** it in MLflow, and set the **`champion`** alias for FastAPI / demos.

**Requires:** Notebook 4 completed so `./mlruns` has runs (with **`als_model`**) under **`Amazon_Product_Recommendation`**. The last cell **exports** the `@champion` model to **`exported_models/als_champion`** for deployment-style bundles, not just the registry pointer.

In [5]:
import os

import mlflow
from mlflow.tracking import MlflowClient

# Same file store as Notebook 4
_mlruns = os.path.abspath(os.path.join(os.getcwd(), "mlruns"))
mlflow.set_tracking_uri("file:" + _mlruns)
client = MlflowClient()
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: file:/Users/bavishna/Documents/Bavishna_Masters/Sem_3/DATA228/Project/Final_project/mlruns


In [7]:
EXPERIMENT_NAME = "Amazon_Product_Recommendation"
MODEL_NAME = "Amazon_ALS_Recommender"
ARTIFACT_SUBPATH = "als_model"

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise RuntimeError(
        f"No experiment '{EXPERIMENT_NAME}'. Run Notebook 4 first."
    )

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.rmse ASC"],
    max_results=50,
)
runs = [r for r in runs if r.data.metrics.get("rmse") is not None]
if not runs:
    raise RuntimeError("No runs with metric 'rmse'. Run Notebook 4 ALS training first.")

best_run = runs[0]
rmse = best_run.data.metrics["rmse"]
print(f"Best run: {best_run.info.run_id}")
print(f"  rmse: {rmse}")
print(f"  run name: {best_run.data.tags.get('mlflow.runName', 'N/A')}")

Best run: cb6fe823ef7d40d4832b6def819786c8
  rmse: 2.0782265038267047
  run name: ALS_Collaborative_Filtering


In [8]:
model_uri = f"runs:/{best_run.info.run_id}/{ARTIFACT_SUBPATH}"
print("Registering from:", model_uri)

result = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
print(f"Registered {MODEL_NAME} version {result.version}")

Registering from: runs:/cb6fe823ef7d40d4832b6def819786c8/als_model
Registered Amazon_ALS_Recommender version 3


Registered model 'Amazon_ALS_Recommender' already exists. Creating a new version of this model...
Created version '3' of model 'Amazon_ALS_Recommender'.


In [9]:
# Point 'champion' at this version (overwrites previous alias if any)
client.set_registered_model_alias(MODEL_NAME, "champion", str(result.version))
print(f"Alias 'champion' -> {MODEL_NAME} v{result.version}")
print(
    "Load in code: models:/Amazon_ALS_Recommender@champion "
    "(or your registry URI if using a remote server)"
)

Alias 'champion' -> Amazon_ALS_Recommender v3
Load in code: models:/Amazon_ALS_Recommender@champion (or your registry URI if using a remote server)


In [10]:
# Export champion artifact to a portable folder (for Docker / offline / deployment bundles)
import shutil

EXPORT_DIR = os.path.abspath(os.path.join("exported_models", "als_champion"))
if os.path.isdir(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)

local_export = mlflow.artifacts.download_artifacts(
    artifact_uri=f"models:/{MODEL_NAME}@champion",
    dst_path=EXPORT_DIR,
)
print("Exported bundle:")
print(" ", local_export)
print("Includes MLmodel + sparkml/ + conda.yaml — load with mlflow.spark.load_model(uri) or ship this directory.")

Exported bundle:
  /Users/bavishna/Documents/Bavishna_Masters/Sem_3/DATA228/Project/Final_project/exported_models/als_champion/
Includes MLmodel + sparkml/ + conda.yaml — load with mlflow.spark.load_model(uri) or ship this directory.


## Where models live

| Location | Role |
|----------|------|
| **`models/als_recommendation_model`** | Written by Notebook 4 — convenient PySpark `load()` path for local demos. |
| **MLflow Registry** (`Amazon_ALS_Recommender` + `@champion`) | Versioned “source of truth” after this notebook’s register/alias steps. |
| **`exported_models/als_champion`** (next cell) | **Filesystem export** of the champion artifact (`MLmodel`, `sparkml/`, env files) for Docker, offline servers, or CI — important when you must ship files outside `mlruns`.